# Baseline: Author's Original TickNet (Model Basic) Benchmarking
## CIFAR-10 & CIFAR-100 Evaluation with SGD lr=0.10 (200 Epochs)

Notebook này chạy mô hình gốc **TickNet Basic** của tác giả (baseline) trên 2 bộ dữ liệu CIFAR-10 và CIFAR-100 để làm mốc đối chứng khoa học với mô hình cải tiến **TickNet-L**.

### Hướng dẫn chạy trên Kaggle:
1. Chọn **Accelerator**: GPU T4 x2 hoặc GPU P100.
2. Bật **Internet**: Always on.
3. Bấm **Save Version** -> **Save & Run All (Commit)**.
4. Tải file `author_ticknet_baseline_results.zip` sau khi chạy xong để làm số liệu đối chiếu trong báo cáo cuối kỳ.


In [ ]:
# 1. Environment & GPU Preflight Check
import os
import torch

# Auto-clone repository from branch feature/final-exam-model-l
if not os.path.exists("train_cifar.py"):
    !test -d /kaggle/working/TickNets/.git || git clone --depth 1 --branch feature/final-exam-model-l https://github.com/khuonglaptri3/TickNets.git /kaggle/working/TickNets
    %cd /kaggle/working/TickNets

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi


In [ ]:
# 2. Download and Verify CIFAR-10 & CIFAR-100 Datasets
!python download_cifar.py --data-root /kaggle/working/data --dataset all


### 3. Huấn Luyện Author TickNet Basic trên CIFAR-10 (SGD lr=0.10, 200 Epochs)

In [ ]:
# Run Baseline CIFAR-10
!python train_cifar.py --config configs/final/baseline_cifar10_sgd_lr010.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/baseline_cifar10_sgd_lr010


### 4. Huấn Luyện Author TickNet Basic trên CIFAR-100 (SGD lr=0.10, 200 Epochs)

In [ ]:
# Run Baseline CIFAR-100
!python train_cifar.py --config configs/final/baseline_cifar100_sgd_lr010.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/baseline_cifar100_sgd_lr010


### 5. Tổng kết Kết Quả Baseline và Đóng Gói Artifacts

In [ ]:
import json
import shutil
from pathlib import Path
import pandas as pd

runs_dir = Path("/kaggle/working/runs")
target_configs = ["baseline_cifar10_sgd_lr010", "baseline_cifar100_sgd_lr010"]
records = []

for exp_name in target_configs:
    p = runs_dir / exp_name
    metric_file = p / "test_metrics.json"
    cfg_file = p / "config.json"
    if metric_file.is_file() and cfg_file.is_file():
        metrics = json.loads(metric_file.read_text())
        cfg = json.loads(cfg_file.read_text())
        records.append({
            "Model": "Author TickNet Basic",
            "Experiment": exp_name,
            "Dataset": cfg["dataset"].upper(),
            "Optimizer": cfg["optimizer"].upper(),
            "LR": cfg["learning_rate"],
            "Test Top-1 (%)": f"{metrics['top1']:.2f}%",
            "Test Loss": f"{metrics['loss']:.4f}",
            "Macro F1": f"{metrics['macro_f1']:.4f}",
            "Params": f"{cfg['learnable_parameters']:,}",
            "FLOPs (G)": f"{cfg['gflops_forward']:.4f}G",
        })

df = pd.DataFrame(records)
print("=== KẾT QUẢ BASELINE TÁC GIẢ ===")
display(df)

# Đóng gói kết quả thành file zip để download
zip_base = Path("/kaggle/working/author_ticknet_baseline_results")
stage_dir = Path("/kaggle/working/stage_baseline_results")
if stage_dir.exists():
    shutil.rmtree(stage_dir)
stage_dir.mkdir(parents=True, exist_ok=True)
for exp in target_configs:
    src = runs_dir / exp
    if src.exists():
        shutil.copytree(src, stage_dir / exp)

shutil.make_archive(str(zip_base), "zip", stage_dir)
print(f"[✓] Đã tạo file kết quả tải về tại: {zip_base}.zip")
